### 2D (no time): peak-list TSV

This section visualizes TSV files that represent a **list of spectral peaks**.

- **TSV schema**: must contain columns **`Frequency (Hz)`** and **`Amplitude`**
- **Inputs**: pass either `files=[...]` (TSV paths, tab-separated) or `dfs=[...]` (pandas DataFrames)
- **Labels**: use `df_labels=[...]` to control legend names (otherwise file path / `DataFrame N` is used)
- **Axis scaling**: `freq_axis_mode` / `amp_axis_mode` mirrors `audiospylt.dft_analysis.analyze_signal()`

In [ ]:
from audiospylt.multiplotter import plot_scatter

# 2D scatter for TSV peak lists.
# Expected columns: 'Frequency (Hz)', 'Amplitude'
# Tip: the axis scaling parameters mirror `audiospylt.dft_analysis.analyze_signal()`.

plot_scatter(
    files=['../data/tables/speech1.tsv', '../data/tables/conga3.tsv'],
    mode='markers',
    # Frequency-axis scaling:
    freq_axis_mode='linear',  # 'linear' | 'log' | 'mel' | 'mixed'
    freq_axis_mix=0.5,        # used only when freq_axis_mode='mixed'
    mixed_log_floor_hz=1.0,   # used only when freq_axis_mode='mixed'
    # Amplitude-axis scaling:
    amp_axis_mode='linear',   # 'linear' | 'log' | 'mixed'
    amp_axis_mix=0.5,         # used only when amp_axis_mode='mixed'
    amp_log_floor=1e-12,      # used for 'log'/'mixed'
    # Figure size:
    plot_width=1100,
    plot_height=700,
    # Range handling:
    auto_plot_range=True,
)

# Sample usage:
# plot_scatter(files=['../data/tables/speech1.tsv', '../data/tables/conga3.tsv'], df_labels=['Speech', 'Conga'], mode='markers')
# plot_scatter(dfs=[df1, df2], df_labels=['DataFrame1', 'DataFrame2'], mode='lines')
# plot_scatter(files=['../data/tables/speech1.tsv'], freq_axis_mode='log', amp_axis_mode='log')

### 2D (time domain): event segments TSV

This section visualizes TSV files where each row is a **time/frequency segment** (drawn as a line between start/stop).

- **TSV schema**: `time_start`, `time_stop`, `freq_start`, `freq_stop`
- **Units**: time in seconds, frequency in Hz (as exported by your analysis pipeline)

In [ ]:
from audiospylt.multiplotter import plot_combined

# 2D visualization for event TSVs that include time/frequency segments.
# Expected columns: 'time_start', 'time_stop', 'freq_start', 'freq_stop'

plot_combined(
    files=['../data/tables/sigmoid_high.tsv', '../data/tables/sigmoid_low.tsv'],
    plot_width=1100,
    plot_height=700,
)

# Sample usage:
# plot_combined(files=['tsv/file1.tsv'])
# plot_combined(dfs=[df2], df_labels=["Label for df2"])

### 3D (time domain): event segments TSV

This section extends the 2D event view into 3D by adding an amplitude dimension.

- **TSV schema**: `time_start`, `time_stop`, `amp_min`, `amp_max`, `freq_start`, `freq_stop`

In [ ]:
from audiospylt.multiplotter import plot_combined_3d

# 3D visualization for "event TSVs" (time, amplitude, frequency).
# Expected columns:
#   - time_start, time_stop
#   - amp_min, amp_max
#   - freq_start, freq_stop
#
# Axis mapping:
#   axis_order=(X, Y, Z) where each is one of: "time" | "amp" | "freq"
#   Examples:
#     axis_order=("time", "amp", "freq")  # X=time, Y=amp, Z=freq (default)
#     axis_order=("freq", "time", "amp")  # X=freq, Y=time, Z=amp
#
# Axis flipping:
#   flip can be given either by dimension name (time/amp/freq) OR by axis name (x/y/z).
#   Axis keys override dimension keys if both are present.
#   Examples:
#     flip={"freq": True}           # flip whichever axis contains frequency
#     flip={"x": True, "z": True}  # flip two axes (X and Z)
#
# Other knobs:
#   axis_pad_frac=0.05      # padding added to computed ranges
#   zero_based_axes=True    # clamps mins to <= 0

plot_combined_3d(
    files=['../data/tables/sigmoid_high.tsv', '../data/tables/sigmoid_low.tsv'],
    axis_order=("time", "amp", "freq"),
    flip={"amp": True},
    plot_width=1100,
    plot_height=700,
)

# Sample usage:
# plot_combined_3d(files=['tsv/file1.tsv', 'tsv/file2.tsv'])
# plot_combined_3d(dfs=[df1, df2], df_labels=['DataFrame1', 'DataFrame2'])


### 2D binned scatter view (density map)

If you want a denser overview than a raw scatter plot, `plot_scatter_binned(...)` groups points into 2D bins and shows them as a histogram-style density map.


In [ ]:
import audiospylt
import audiospylt.multiplotter as mp

print(audiospylt.__file__)
print(mp.__file__)

In [ ]:
from audiospylt.multiplotter import plot_scatter_binned

# Binned 2D view for peak-list TSVs.
# Points are aggregated into frequency/amplitude bins instead of being drawn individually.

plot_scatter_binned(
    files=['../data/tables/speech1.tsv', '../data/tables/conga3.tsv'],
    x_bin_size=20.0,
    y_bin_size=0.002,
    freq_axis_mode='linear',
    amp_axis_mode='linear',
    title='Binned scatter: speech vs conga',
    plot_width=1100,
    plot_height=700,
)


### Equalizer-style bar view (custom bin sizes)

If you have lots of points, an equalizer-like binned bar view is often easier to read than a raw scatter.

- **What it does**: bins frequencies into fixed-width **Hz** bins and aggregates amplitudes per bin (e.g. `agg='max'`).
- **Per-TSV colors**: each TSV/DataFrame becomes its own trace with a unique color.
- **Axis scaling**: `freq_axis_mode` / `amp_axis_mode` are supported here too.
- **Note on bin size**: `x_bin_size` is in **Hz before any optional frequency warping** (mel/mixed) is applied for display.


In [ ]:
from audiospylt.multiplotter import plot_equalizer_bars

# Equalizer-like bars: frequencies are grouped into fixed-width Hz bins.
# Each TSV becomes its own colored bar trace.
# Axis scaling options mirror `audiospylt.dft_analysis.analyze_signal()`.

plot_equalizer_bars(
    files=['../data/tables/speech1.tsv', '../data/tables/conga3.tsv'],
    x_bin_size=10.0,   # Hz per bar
    agg='max',         # 'max' | 'sum' | 'mean'
    mode='overlay',    # 'group' | 'overlay' | 'stack'
    opacity=0.75,
    # Frequency-axis scaling:
    freq_axis_mode='linear',  # 'linear' | 'log' | 'mel' | 'mixed'
    freq_axis_mix=0.5,        # used only when freq_axis_mode='mixed'
    mixed_log_floor_hz=1.0,   # used only when freq_axis_mode='mixed'
    # Amplitude-axis scaling:
    amp_axis_mode='linear',   # 'linear' | 'log' | 'mixed'
    amp_axis_mix=0.5,         # used only when amp_axis_mode='mixed'
    amp_log_floor=1e-12,      # used for 'log'/'mixed'
    # Figure size:
    plot_width=1100,
    plot_height=700,
)
